# Task 2 — corruption classifier and hard-routed specialists
Upload this notebook into Colab, select **Runtime → Change runtime type → GPU**, then **Run all**. Upload `task2_source.zip` when requested and allow Drive access.

The classifier predicts clean, salt-and-pepper, Gaussian blur or rectangular occlusion. Training batches contain exactly equal numbers of each class, with four runtime views per training photo. Three independent compressed autoencoders train exclusively on their own corruption; predicted-clean images use an exact identity bypass. The router receives no corruption label at inference.

Each component gets a separate Optuna study. Starting budgets: classifier 8 trials × 5 epochs then 20 final epochs; each specialist 6 trials × 8 epochs then 60 final epochs. Budgets are editable below before starting. Final training starts from scratch using the selected configuration.

All files use `/content/ass1_task2` and `MyDrive/GenAI_Ass1/task2_v1`. Checkpoints are backed up after each completed epoch. On reconnect, rerun the notebook with the same settings to restore studies and resume final training. The final cell downloads **task2_results.zip**.

This notebook evaluates validation only: the original fixed 408 cases and a separate 4,080-case severity sweep. It exports all four ONNX components and verifies the actual hard-routing runtime against PyTorch. No restoration improvement is assumed before these results.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, zipfile, shutil, subprocess, sys, json, sqlite3
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime, then reconnect before training.')
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PROJECT = Path('/content/ass1_task2')
BACKUP = Path('/content/drive/MyDrive/GenAI_Ass1/task2_v1')
PROJECT.mkdir(exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)


In [ ]:
uploaded = files.upload()  # Select colab/task2_source.zip
if len(uploaded) != 1:
    raise ValueError('Upload exactly the generated source ZIP.')
archive_name = next(iter(uploaded))
with zipfile.ZipFile(archive_name) as archive:
    for entry in archive.infolist():
        destination = (PROJECT / entry.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError('Invalid ZIP path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
# Keep Colab's CUDA-enabled PyTorch installation; do not install the CPU wheel.
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

config = json.loads(Path('configs/task2.json').read_text())
print('Task 2 configuration:', json.dumps(config, indent=2))


In [ ]:
# Restore previous results after reconnecting, and define consistent backup helpers.
for folder in ['runs', 'models', 'mlruns']:
    if (BACKUP / folder).exists():
        shutil.copytree(BACKUP / folder, PROJECT / folder, dirs_exist_ok=True)
if (BACKUP / 'mlflow.db').exists():
    shutil.copy2(BACKUP / 'mlflow.db', PROJECT / 'mlflow.db')

def backup_results():
    ignore = shutil.ignore_patterns('*.sqlite3', '*.db', '*.tmp', '*-journal', '*-wal', '*-shm')
    for folder in ['runs', 'models', 'mlruns']:
        if (PROJECT / folder).exists():
            shutil.copytree(PROJECT / folder, BACKUP / folder, dirs_exist_ok=True, ignore=ignore)
    databases = list((PROJECT / 'runs').rglob('*.sqlite3')) if (PROJECT / 'runs').exists() else []
    if (PROJECT / 'mlflow.db').exists():
        databases.append(PROJECT / 'mlflow.db')
    for source in databases:
        local_copy = Path('/content') / (source.name + '.backup')
        local_copy.unlink(missing_ok=True)
        with sqlite3.connect(str(source), timeout=60) as src, sqlite3.connect(str(local_copy)) as dest:
            src.backup(dest)
        destination = BACKUP / source.relative_to(PROJECT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_copy, destination)

def run(arguments):
    process = subprocess.Popen([sys.executable, '-u', *arguments], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, cwd=PROJECT)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(record, dict) and 'epoch' in record:
                backup_results()
        if process.wait() != 0:
            raise RuntimeError('Command failed; inspect the output above.')
    except BaseException:
        process.terminate()
        process.wait()
        backup_results()
        raise
    backup_results()


In [ ]:
# Download data to local runtime storage; preserve official split definitions.
import urllib.request, tarfile
raw = PROJECT / 'raw_data'
raw.mkdir(exist_ok=True)
for filename in ['102flowers.tgz', 'setid.mat']:
    path = raw / filename
    if not path.exists():
        temporary = raw / (filename + '.part')
        urllib.request.urlretrieve('https://www.robots.ox.ac.uk/~vgg/data/flowers/102/' + filename, temporary)
        temporary.replace(path)
if not (raw / 'jpg' / 'image_08189.jpg').exists():
    with tarfile.open(raw / '102flowers.tgz') as archive:
        archive.extractall(raw, filter='data')
run(['prepare_dataset.py'])
# Fail before training if preparation differs from the original trained baseline.
import hashlib
baseline_data = json.loads(Path('experiments/task2/baseline_data.json').read_text())
for filename, key in [('train.json', 'train_manifest_sha256'),
                      ('validation_manifest.json', 'validation_manifest_sha256')]:
    manifest = Path('data/prepared_v2/restoration') / filename
    assert hashlib.sha256(manifest.read_bytes()).hexdigest() == baseline_data[key], filename + ' differs from baseline'
print('Training and validation manifests match the baseline.')
run(['-m', 'unittest', 'discover', '-s', 'tests', '-v'])


## Configure the training budget
One classifier epoch uses four views per training photo; one specialist epoch uses one view per photo. Smoke limits are deliberately absent here so downloaded checkpoints are full-data training runs.


In [ ]:
import optuna
ROOT = Path('runs/task2')
CLASSIFIER_TRIALS = 8
CLASSIFIER_TRIAL_EPOCHS = 5
SPECIALIST_TRIALS = 6
SPECIALIST_TRIAL_EPOCHS = 8
CLASSIFIER_EPOCHS = 20
SPECIALIST_EPOCHS = 60
# These are starting budgets, not an exhaustive optimization study.
# Keep the same budgets/configuration when resuming an existing study.

def train_component(component):
    is_classifier = component == 'classifier'
    search = ROOT / component / 'search'
    final = ROOT / component / 'final'
    budget = CLASSIFIER_TRIALS if is_classifier else SPECIALIST_TRIALS
    trial_epochs = CLASSIFIER_TRIAL_EPOCHS if is_classifier else SPECIALIST_TRIAL_EPOCHS
    final_epochs = CLASSIFIER_EPOCHS if is_classifier else SPECIALIST_EPOCHS
    study_path = search / 'study.sqlite3'
    finished = 0
    if study_path.exists():
        study = optuna.load_study(study_name='task2-' + component,
                                 storage='sqlite:///' + str(study_path.resolve()))
        finished = sum(t.state in [optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED]
                       for t in study.trials)
    remaining = max(0, budget - finished)
    if remaining:
        run(['-m', 'task2.train', 'tune', '--component', component, '--config', 'configs/task2.json',
             '--epochs', str(trial_epochs), '--trials', str(remaining), '--device', 'cuda',
             '--workers', '2', '--output', str(search)])
    selected = search / 'best_config.json'
    chosen = json.loads(selected.read_text())
    chosen['epochs'] = final_epochs
    last = final / 'last.pt'
    completed = 0
    if last.exists():
        saved = torch.load(last, map_location='cpu', weights_only=True)
        assert saved['config'] == chosen, 'Selected config changed; use a new experiment folder.'
        completed = saved['epoch']
    if completed < final_epochs:
        arguments = ['-m', 'task2.train', 'train', '--component', component, '--config', str(selected),
                     '--epochs', str(final_epochs), '--device', 'cuda', '--workers', '2', '--output', str(final)]
        if last.exists():
            arguments += ['--resume', str(last)]
        run(arguments)
    else:
        print(component, 'already completed', completed, 'epochs')
    print('Finished', component, 'using', chosen)
    backup_results()


## Four-class corruption classifier


In [ ]:
train_component('classifier')


## Salt-and-pepper specialist


In [ ]:
train_component('salt_and_pepper')


## Gaussian blur specialist


In [ ]:
train_component('gaussian_blur')


## Occlusion specialist


In [ ]:
train_component('rectangular_occlusion')


## Oracle versus predicted routing
The original validation manifest gives a direct condition-wise comparison with the Task 1 Run A model. The severity sweep tests mild/medium/severe corruption on every validation photo. Inspect clean predictions, confusion matrices, misrouting penalties and restoration failures separately. Exact clean identity yields capped PSNR of 120 dB, so the overall mean can be misleading.


In [ ]:
run(['-m', 'task2.evaluate', '--checkpoints', str(ROOT), '--split', 'val',
     '--device', 'cuda', '--output', 'runs/task2/validation'])
run(['-m', 'task2.evaluate', '--checkpoints', str(ROOT), '--split', 'val', '--severity-sweep',
     '--device', 'cuda', '--output', 'runs/task2/severity_validation'])
from IPython.display import display, Image
for folder in ['validation', 'severity_validation']:
    for name in ['confusion_matrix.png', 'routing_comparison.png']:
        display(Image(filename=str(ROOT / folder / name)))
for path in sorted((ROOT / 'severity_validation').glob('routing_failure_*.png')):
    display(Image(filename=str(path)))


In [ ]:
import pandas as pd
current = json.loads((ROOT / 'validation/results.json').read_text())
previous = json.loads(Path('experiments/task2/universal_run_a_validation.json').read_text())
assert current['metrics'].keys() == previous['metrics'].keys()
rows = []
for group, scores in current['metrics'].items():
    baseline = previous['metrics'][group]
    assert scores['n'] == baseline['n']
    rows.append({'group': group, 'n': scores['n'], 'routing_accuracy': scores['routing_accuracy'],
                 'input_psnr': scores['input_psnr'], 'universal_run_a_psnr': baseline['psnr'],
                 'oracle_psnr': scores['oracle_psnr'], 'predicted_psnr': scores['predicted_psnr'],
                 'universal_run_a_ssim': baseline['ssim'], 'oracle_ssim': scores['oracle_ssim'],
                 'predicted_ssim': scores['predicted_ssim']})
comparison = pd.DataFrame(rows)
comparison.to_csv(ROOT / 'task1_comparison.csv', index=False)
display(comparison.round(4))
backup_results()


## Export and verify deployment
Four fixed batch-one, RGB 128×128 ONNX models: classifier probabilities and three specialist reconstructions. Class order and artifact hashes are recorded in pipeline.json. The following check uses the same ONNX router as FastAPI and covers each ground-truth condition; actual predicted routes depend on the trained classifier.


In [ ]:
run(['-m', 'task2.export', '--checkpoints', str(ROOT), '--output', 'models/task2'])
from backend.task2_runtime import Task2Runtime
from task2.model import load_pipeline
from dataset_loaders import RestorationDataset
import numpy as np
runtime = Task2Runtime('models/task2')
pytorch, _ = load_pipeline(ROOT)
with torch.no_grad():
    for condition in ['clean','salt_and_pepper','gaussian_blur','rectangular_occlusion']:
        dataset = RestorationDataset(split='val', condition=condition, limit=4)
        for item in dataset:
            x = item['input'][None]
            expected, expected_probabilities, expected_route = pytorch(x)
            actual, probabilities, route = runtime.run(x.numpy())
            np.testing.assert_allclose(actual, expected.numpy(), rtol=1e-4, atol=1e-5)
            np.testing.assert_allclose(probabilities, expected_probabilities.numpy()[0], rtol=1e-4, atol=1e-5)
            assert route == int(expected_route[0])
print('Hard-routed ONNX runtime parity passed on 16 fixed validation cases.')
backup_results()


## Download trained models and evidence


In [ ]:
backup_results()
output = Path('/content/task2_results.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ['models', 'runs', 'mlruns', 'experiments/task2']:
        for path in (PROJECT / folder).rglob('*'):
            if path.is_file() and not path.name.endswith(('-journal','-wal','-shm','.tmp','.sqlite3')):
                archive.write(path, path.relative_to(PROJECT))
            elif path.is_file() and path.suffix == '.sqlite3':
                archive.write(BACKUP / path.relative_to(PROJECT), path.relative_to(PROJECT))
    archive.write(PROJECT / 'configs/task2.json', 'configs/task2.json')
    if (BACKUP / 'mlflow.db').exists():
        archive.write(BACKUP / 'mlflow.db', 'mlflow.db')
files.download(str(output))
